# 1. Basic Concepts

In [2]:
import simpy

# Define the "car" process
def car(env):
    while True:
        print('Start parking at %d' % env.now)
        parking_duration = 5
        yield env.timeout(parking_duration)

        print('Start driving at %d' % env.now)
        trip_duration = 2
        yield env.timeout(trip_duration)

# Create environment
env = simpy.Environment()

# Enroll "car" process in the environment
env.process(car(env))

# "until" is the stop criteria, which can be a time duration or an event
env.run(until=15)

Start parking at 0
Start driving at 5
Start parking at 7
Start driving at 12
Start parking at 14


# 2. Process Interaction

## 2.1. Waiting for a Process

In [16]:
class Car(object):
    def __init__(self,env):
        self.env = env
        # Start the run process everytime an instance is created.
        self.action = env.process(self.run())

    def run(self):
        while(True):
            print("Starting parking and charging at %d" %self.env.now)
            charge_duration = 5
            
            try: 
                yield self.env.process(self.charge(charge_duration))
            except simpy.Interrupt:
                # When we received an interrupt, we stop charging and
                # switch to the "driving" state
                print('Was interrupted. Hope, the battery is full enough ...')

            print("Starting driving at %d" %self.env.now)
            trip_duration = 2
            yield self.env.timeout(trip_duration)

    def charge(self,charge_duration):
        yield self.env.timeout(charge_duration)

In [17]:
import simpy
env = simpy.Environment()
car = Car(env)
env.run(until=15)

Starting parking and charging at 0
Starting driving at 5
Starting parking and charging at 7
Starting driving at 12
Starting parking and charging at 14


## 2.2. Interrupting another Process

In [18]:
def driver(env, car):
    yield env.timeout(3)
    car.action.interrupt()

env = simpy.Environment()
car = Car(env)
env.process(driver(env, car))
env.run(until=15)

Starting parking and charging at 0
Was interrupted. Hope, the battery is full enough ...
Starting driving at 3
Starting parking and charging at 5
Starting driving at 10
Starting parking and charging at 12


# 3. Shared Resource Usage

## 3.1. Basic Resource Usage

The car will now drive to a battery charging station (BCS) and request one of its two charging spots. If both of these spots are currently in use, it waits until one of them becomes available again. It then starts charging its battery and leaves the station afterwards:

In [48]:
def car(env, name, bcs, driving_time, charge_duration):
    # Simulate driving to the BCS
    yield env.timeout(driving_time)

    # Request one of its charging spots
    print('%s arriving at %d' % (name, env.now))

    # Resource's "request()" method lets you wait until the resource is available
    # If you use the resource with the with statement as shown above, the resource is automatically being released.
    # If you call request() without with, you are responsible to call release() once you are done using the resource.

    with bcs.request() as request:
        yield request

        # Charge the battery
        print('%s starting to charge at %s' % (name, env.now))
        yield env.timeout(charge_duration)


        print('%s leaving the bcs at %s' % (name, env.now))

In [49]:
import simpy
env = simpy.Environment()

# A resource has a "capacity" feature
# The basic "Resource" sorts waiting processes in FIFO
bcs = simpy.Resource(env, capacity=2) 

In [50]:
# Creating 4 cars
for i in range(4):
    env.process(car(env, 'Car %d' % i, bcs, i*2, 5))

In [ ]:
env.run()

Car 0 arriving at 0
Car 0 starting to charge at 0
Car 1 arriving at 2
Car 1 starting to charge at 2
Car 2 arriving at 4
Car 0 leaving the bcs at 5
Car 2 starting to charge at 5
Car 3 arriving at 6
Car 1 leaving the bcs at 7
Car 3 starting to charge at 7


In [59]:
def producer(env, store):
    for i in range(100):
        yield env.timeout(2)
        yield store.put(f'item {i}')
        print(f't={env.now}: Produced item {i}')


def consumer(name, env, store):
    while True:
        yield env.timeout(1)
        print(f't={env.now}: {name} requesting item')

        item = yield store.get()

        print(f't={env.now}: {name} got {item}')


env = simpy.Environment()
store = simpy.Store(env, capacity=2)

prod = env.process(producer(env, store))
consumers = [
    env.process(consumer(f"Consumer {i}", env, store))
    for i in range(2)
]

env.run(until=5)

t=1: Consumer 0 requesting item
t=1: Consumer 1 requesting item
t=2: Produced item 0
t=2: Consumer 0 got item 0
t=3: Consumer 0 requesting item
t=4: Produced item 1
t=4: Consumer 1 got item 1


In [64]:
import time
import simpy

def example(env):
    start = time.perf_counter()
    yield env.timeout(1)
    end = time.perf_counter()
    print('Duration of one simulation time unit: %.2fs' % (end - start))

env = simpy.Environment()
proc = env.process(example(env))
env.run(until=proc)

import simpy.rt
env = simpy.rt.RealtimeEnvironment(factor=3)
proc = env.process(example(env))
env.run(until=proc)

Duration of one simulation time unit: 0.00s
Duration of one simulation time unit: 3.00s


(0, 0, <class 'simpy.events.Initialize'>)
(1, 1, <class 'simpy.events.Timeout'>)
(1, 2, <class 'simpy.events.Process'>)
